# sPHENIX Module 1 — Exercises
**Scope:** Reader Week 1 + Module 1 (lessons 1.1–1.8). Five exercises, each with a short background. Exercises 2, 4, and 5 are runnable right here in WSL; 1 and 3 are answer-in-markdown / on-cluster tasks.

Work top to bottom. Hints are collapsed under each exercise — try first, peek second.


---
## Exercise 1 — Environment forensics 🩺

**Background.** Every sPHENIX login starts with `source /opt/sphenix/core/bin/sphenix_setup.sh -n new`, which populates variables like `$OFFLINE_MAIN` (the release dir) and `$ROOTSYS`. A second script, `setup_local.sh $MYINSTALL`, prepends *your* install dir to `LD_LIBRARY_PATH` and `ROOT_INCLUDE_PATH` so the framework can load libraries **you** build. The #1 new-student failure mode is a broken or half-sourced environment.

**Task.** For each scenario, write (in the markdown cell below) what is wrong and the *first command you'd run* to confirm it:

1. You log in, type `root`, and get `bash: root: command not found`.
2. ROOT starts fine, but your Fun4All macro fails with `error loading libMyAnalysis.so`. Yesterday it worked. (You rebuilt nothing.)
3. Your macro runs but reads DSTs produced with `ana.464` while your shell is on `-n new`, and a node it needs is mysteriously missing.
4. A colleague's script starts with `#!/bin/bash` and immediately calls `root -l -b -q macro.C` — it works in their terminal but dies under Condor. What line(s) are missing?


**Your answers:**

1.
2.
3.
4.

<details><summary>Hints / solution sketch</summary>

1. Setup never sourced → `echo $OFFLINE_MAIN` (empty). Fix: source `sphenix_setup.sh`.
2. `setup_local.sh` not sourced this session → `echo $MYINSTALL` / check `echo $LD_LIBRARY_PATH | tr ':' '\n' | grep install`.
3. Build/data mismatch — analyze `ana.464` DSTs with `ana.464`, not `new`. Confirm with `echo $OFFLINE_MAIN`.
4. Scripts get a fresh shell: they must source `sphenix_setup.sh` (+ `setup_local.sh`) themselves before calling `root`.
</details>


---
## Exercise 2 — Linux skills drill (runnable here) 🐧

**Background.** On the cluster you'll constantly build directory trees with brace expansion (`mkdir -p a/{b,c}`), hunt files with `find`, search code with `grep -r`, and glue it together in small bash scripts beginning with `set -euo pipefail`. This drill builds those reflexes locally in WSL — same commands, zero risk.

**Task.** In the cell below:

1. Create a sandbox tree `~/m1_drill/{scripts,data,plots,logs}` with one command.
2. Generate five fake macro files `ana_0.C … ana_4.C` in `scripts/`, each containing a few lines (your choice — e.g. a loop writing `// line N`).
3. Use `find` to list every `.C` file under `~/m1_drill`.
4. Use `grep -rn` to find which of your files contain the string `line 2`.
5. Write (and run) a loop that prints `<filename>: <line count>` for every `.C` file — the Lesson 1.5 drill, item 4.


In [ ]:
%%bash
set -euo pipefail
# 1. sandbox tree

# 2. fake macros

# 3. find all .C files

# 4. grep for "line 2"

# 5. loop printing "<file>: <line count>"


<details><summary>Hints / solution sketch</summary>

```bash
mkdir -p ~/m1_drill/{scripts,data,plots,logs}
for i in 0 1 2 3 4; do
  for n in 1 2 3; do echo "// line $n" >> ~/m1_drill/scripts/ana_${i}.C; done
done
find ~/m1_drill -name "*.C"
grep -rn "line 2" ~/m1_drill/scripts/
for f in ~/m1_drill/scripts/*.C; do echo "$f: $(wc -l < "$f")"; done
```
</details>


---
## Exercise 3 — The sPHENIX git ritual 🌿

**Background.** sPHENIX work follows one loop: update `master` → **branch** → edit → `add`/`commit` → `push` → PR. Committing on `master`, force-pushing shared branches, or committing `.root` files are the three cardinal sins. This exercise rehearses the loop in a local scratch repo (swap in the real `analysis` repo once you're on SDCC).

**Task.** In the cell below (or a terminal):

1. `git init` a scratch repo `~/m1_gitdrill`, add a first commit on `master`/`main` (e.g. a `README.md`).
2. Create branch `sandbox/yourname`.
3. Add `NOTES.md` with one line, commit with a *descriptive* message.
4. Show `git log --oneline -5` and confirm your commit is on top.
5. Simulate review feedback: edit `NOTES.md`, make a **second** commit on the same branch (this is how you respond to PR comments — never force-push).
6. Bonus: run `git checkout master` then `git merge sandbox/yourname` and look at the log graph: `git log --all --graph --oneline`.


In [ ]:
%%bash
set -euo pipefail
rm -rf ~/m1_gitdrill   # fresh start each run
# 1. init + first commit

# 2. branch

# 3. NOTES.md + commit

# 4. log

# 5. second commit (review response)

# 6. bonus: merge + graph


<details><summary>Hints / solution sketch</summary>

```bash
mkdir ~/m1_gitdrill && cd ~/m1_gitdrill && git init
echo "# drill" > README.md && git add README.md && git commit -m "Initial commit"
git checkout -b sandbox/plewis
echo "Week 1 notes live here" > NOTES.md
git add NOTES.md && git commit -m "Add sandbox notes file for Week 1 drill"
git log --oneline -5
echo "Addressed review: added detail" >> NOTES.md
git add NOTES.md && git commit -m "Expand notes after review feedback"
git checkout master 2>/dev/null || git checkout main
git merge sandbox/plewis
git log --all --graph --oneline
```
</details>


---
## Exercise 4 — Histograms three ways (course Ex. 1.7) 📊

**Background.** The ROOT workflow is: book a histogram (`TH1F("name", "title;x;y", nbins, lo, hi)`), `Fill()` it in a loop, `Draw()` on a `TCanvas`, optionally `Fit("gaus")`, then `SaveAs`. Overlaying histograms uses `Draw("SAME")` plus a `TLegend`. Since ROOT isn't installed in this WSL env, do part A as a macro to run on SDCC (or any ROOT install), and part B as a runnable Python mirror so you can check the picture *now*.

**Task A (ROOT macro — write it, run on cluster).** Write `myFirst.C` that:
1. Fills three `TH1F`s: Gaussian(μ=5, σ=1), Gaussian(μ=5, σ=2), Uniform on [0,10] — 100k entries each.
2. Draws all three on one canvas in different colors (`SetLineColor`), widest-first so nothing is clipped.
3. Adds a `TLegend` labeling each.
4. Saves to `myFirst.pdf`.

**Task B (runnable now).** Reproduce the same three distributions + overlay in the Python cell below and confirm the shapes look right.


In [ ]:
%%writefile myFirst.C
// Task A — fill in the blanks, then run on SDCC:  root -l -b -q myFirst.C
void myFirst() {
    TH1F *hg1 = new TH1F("hg1", "Three distributions;x;counts", 100, 0, 10);
    TH1F *hg2 = /* Gaussian sigma=2 */ nullptr;
    TH1F *hu  = /* Uniform [0,10]   */ nullptr;

    for (int i = 0; i < 100000; ++i) {
        // hg1->Fill(gRandom->Gaus(5, 1));
        // ... fill hg2 and hu (hint: gRandom->Uniform(0,10))
    }

    TCanvas *c = new TCanvas("c", "", 800, 600);
    // colors, Draw / Draw("SAME"), TLegend, SaveAs("myFirst.pdf")
}

In [ ]:
# Task B — Python mirror (runs now)
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
g1 = rng.normal(5, 1, 100_000)
g2 = rng.normal(5, 2, 100_000)
u  = rng.uniform(0, 10, 100_000)

bins = np.linspace(0, 10, 101)
plt.hist(g2, bins=bins, histtype="step", color="red",   label=r"Gaus($\mu$=5, $\sigma$=2)")
plt.hist(g1, bins=bins, histtype="step", color="blue",  label=r"Gaus($\mu$=5, $\sigma$=1)")
plt.hist(u,  bins=bins, histtype="step", color="green", label="Uniform [0,10]")
plt.xlabel("x"); plt.ylabel("counts"); plt.legend()
plt.title("Exercise 4 — three distributions")
plt.show()

<details><summary>Hints / ROOT solution sketch</summary>

```cpp
void myFirst() {
    TH1F *hg1 = new TH1F("hg1", "Three distributions;x;counts", 100, 0, 10);
    TH1F *hg2 = new TH1F("hg2", "", 100, 0, 10);
    TH1F *hu  = new TH1F("hu",  "", 100, 0, 10);
    for (int i = 0; i < 100000; ++i) {
        hg1->Fill(gRandom->Gaus(5, 1));
        hg2->Fill(gRandom->Gaus(5, 2));
        hu->Fill(gRandom->Uniform(0, 10));
    }
    hg1->SetLineColor(kBlue); hg2->SetLineColor(kRed); hu->SetLineColor(kGreen+2);
    TCanvas *c = new TCanvas("c", "", 800, 600);
    hg1->Draw(); hg2->Draw("SAME"); hu->Draw("SAME");
    auto leg = new TLegend(0.65, 0.7, 0.88, 0.88);
    leg->AddEntry(hg1, "Gaus(5,1)", "l");
    leg->AddEntry(hg2, "Gaus(5,2)", "l");
    leg->AddEntry(hu,  "Uniform",   "l");
    leg->Draw();
    c->SaveAs("myFirst.pdf");
}
```
Bonus from the course: a second macro that re-opens the output and re-fits with `h->Fit("gaus")`.
</details>


---
## Exercise 5 — C++ self-assessment: the `Particle` class (course Ex. 1.8, runnable) ⚙️

**Background.** Every sPHENIX module is a C++ class; analysis code lives in loops over STL containers of objects with getters (`track->get_pt()`). This exercise is the course's fluency gate: a standalone program (no ROOT) with a class, private members, a `std::vector`, and a cut in a loop. Course standard: *smooth in under 20 minutes = ready; otherwise spend a day on learncpp.com.* `g++` is available in WSL, so this runs end-to-end right here.

**Task.** Complete `test.cc` below so it:
1. Defines class `Particle` with **private** `pt, eta, phi` and public getters.
2. Builds a `std::vector<Particle>` of 10 random particles (pt ∈ [0,10], eta ∈ [−2,2], phi ∈ [−π,π]).
3. Loops and prints `pt` for particles with `|eta| < 1`.
4. Compiles with `g++ -std=c++17` and runs (second cell).


In [ ]:
%%writefile test.cc
#include <iostream>
#include <vector>
#include <random>
#include <cmath>

class Particle {
    // 1. private members pt, eta, phi + a constructor + public getters
};

int main() {
    std::mt19937 rng(7);
    std::uniform_real_distribution<float> d_pt(0, 10), d_eta(-2, 2), d_phi(-M_PI, M_PI);

    std::vector<Particle> particles;
    // 2. fill with 10 random particles

    // 3. print pt for |eta| < 1

    return 0;
}

In [ ]:
%%bash
g++ -std=c++17 -o test test.cc && ./test

<details><summary>Hints / solution sketch</summary>

```cpp
class Particle {
 public:
    Particle(float pt, float eta, float phi) : m_pt(pt), m_eta(eta), m_phi(phi) {}
    float get_pt()  const { return m_pt; }
    float get_eta() const { return m_eta; }
    float get_phi() const { return m_phi; }
 private:
    float m_pt, m_eta, m_phi;
};

// in main():
for (int i = 0; i < 10; ++i)
    particles.emplace_back(d_pt(rng), d_eta(rng), d_phi(rng));

for (const auto& p : particles)
    if (std::abs(p.get_eta()) < 1)
        std::cout << "pt = " << p.get_pt() << '\n';
```
Note the sPHENIX naming convention: `m_` prefix for member variables, `get_x()` getters — same style as `SvtxTrack::get_px()`.
</details>

---
**Done?** Check yourself against the Module 1 quiz in the notes notebook, then move to Module 2 (Fun4All).
